In [ ]:
# 0
#!pip install numpy_financial

In [ ]:
# 1
###################### Librerías y acceso ######################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#import numpy_financial as npf
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

# Accedo a la planilla
spreadsheet = gc.open('Copia de Modelo GF v9')
worksheet = spreadsheet.worksheet('Parámetros Técnicos')

In [ ]:
# 2
global tope_capacidad_transporte, vida_util_renovada, vida_util_actual, vida_util_mejorada, vida_util_resto, vida_util_porc
global valor_primera_renovacion, valor_primer_mejoramiento1, anios_obra_renovacion, anios_obra_mejoramiento1, rejuve_obra_mejoramiento1
global consumo_vida_util, consumo_vida_util_actual, disparo_obra_renovacion, disparo_obra_mejoramiento1, disparo_obra_mejoramiento2
global duracion_anios_negocio, tasa_descuento, crec_factor_inicial, periodo_anios_crec_inicial, crec_factor_final, canon_comercial
global costo_desvio, costo_fijo_conservacion, costo_manten_inicial, costo_manten1, costo_manten_renov, carga_ref_manten, tope_inicial_manten
global crec_manten, mantenimiento_renovacion, carga_ref_renov, tope_inicial_renov, crec_renov, repe_obra_renov, repe_obra_mejor1, repe_obra_mejor2
global tipo_via_inicio, tipo_trocha, crec_factor_inicial, crec_factor_final, disparo_obra_mejoramiento1
global cubi_pan1, cubi_pan2, cubi_pan3, carga_ref_pan1, carga_ref_pan2, repe_obra_mejor2
global valor_primer_mejoramiento2, anios_obra_mejoramiento2, rejuve_obra_renov, marca_mejoramiento1, marca_mejoramiento2, marca_renovacion
global rejuve_obra_mejoramiento1, rejuve_obra_mejoramiento2, disparo_obra_mejoramiento2
global pan_carga_ref1, pan_carga_ref2, pan_cubi1, pan_cubi2, pan_cubi3, costo_obra_dict
global id_tramo, consumo_vida_util_actual, tipo_via_inicio, carga_util_teorica, tipo_trocha, barreras, long_tramo

In [ ]:
# 4
####################### Función que Lee las variables del modelo ###############

def get_variables_as_dict(spreadsheet_name, sheet_name='Variables del Modelo'):
  """
  Reads a specified sheet from a Google Spreadsheet and returns a dictionary
  where keys are variable names from the third column and values are the
  corresponding values from the second column.

  Args:
    spreadsheet_name: The name of the Google Spreadsheet.
    sheet_name: The name of the sheet to read.

  Returns:
    A dictionary containing variable names and their values.
  """
  spreadsheet = gc.open(spreadsheet_name)
  worksheet = spreadsheet.worksheet(sheet_name)

  # Get all the values from the worksheet as a list of lists
  data = worksheet.get_all_values()

  variables_dict = {}

  # Start from row 1 to skip headers
  for row in data[1:]:
    # Check if the row has at least 3 columns
    if len(row) >= 3:
      variable_name = row[2].strip() # Variable name is in the third column (index 2)
      cell_value = row[1].strip()   # Value is in the second column (index 1)
      variable_value = cell_value.replace(',', '.', 1) # Handle potential comma as decimal separator

      if variable_name: # Only process if variable name is not empty
        try:
          # Attempt to convert to a more specific type if possible
          if variable_value.lower() == 'true':
              variables_dict[variable_name] = True
          elif variable_value.lower() == 'false':
              variables_dict[variable_name] = False
          elif variable_value.isdigit():
              variables_dict[variable_name] = int(variable_value)
          else:
              try:
                  variables_dict[variable_name] = float(variable_value)
              except ValueError:
                  variables_dict[variable_name] = variable_value # Keep as string if conversion fails
          # print(f"Read variable: {variable_name} = {variables_dict[variable_name]}") # Optional: for debugging
        except Exception as e:
          print(f"Could not process row for variable '{variable_name}': {e}")
    # else:
      # print(f"Skipping row with insufficient columns: {row}") # Optional: for debugging

  return variables_dict

# Example usage with the corrected function:
model_variables = get_variables_as_dict('Copia de Modelo GF v9', 'Variables del Modelo')

# Print the resulting dictionary
#print (model_variables)

for var_name, var_value in model_variables.items():
   globals()[var_name] = var_value
   print(var_name, var_value)

# Now you can access the variables directly, for example:
# print(duracion_anios_negocio)
# print(tasa_descuento)
# print(canon_comercial)

print (duracion_anios_negocio)

duracion_anios_negocio 40
tasa_descuento 0.1
crec_factor_inicial 0.05
periodo_anios_crec_inicial 10
crec_factor_final 0.03
canon_comercial 0.03
costo_pct SI
40


In [ ]:
# 5

#### Función para leer los tramos, hay algunos definidos explicitamente como str
# tipo_via y tipo_trocha son str

def leer_listado_tramos(spreadsheet_name, sheet_name="Listado de Tramos"):
  """
  Reads the 'Listado de Tramos' sheet from a Google Spreadsheet,
  considering row 1 as headers and data starting from row 3.

  Args:
    spreadsheet_name: The name of the Google Spreadsheet.
    sheet_name: The name of the sheet to read (defaults to 'Listado de Tramos').

  Returns:
    A pandas DataFrame containing the data from the specified sheet.
  """
  spreadsheet = gc.open(spreadsheet_name)
  worksheet = spreadsheet.worksheet(sheet_name)

  # Get all the values from the worksheet as a list of lists
  data = worksheet.get_all_values()

  # The column names are in row 1 (index 0). The data starts from row 3 (index 2).
  header = data[0]
  data_rows = data[1:]

  df = pd.DataFrame(data_rows, columns=header)

  # Identify columns to convert
  # All columns except 'tipo_trocha' and those starting with 'tipo_via'
  cols_to_convert = [col for col in df.columns if col != 'tipo_trocha' and not col.startswith('tipo_via')]

  # Convert comma decimals to points and then to numeric, handling errors
  for col in cols_to_convert:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce') # Coerce will turn unparseable values into NaN
  return df

# Example usage:
#df_tramos_from_function = leer_listado_tramos('Copia de Modelo GF v9')
#print(df_tramos_from_function.head())
#print(df_tramos_from_function.info()) # Check the data types of the columns


In [ ]:
## Sólo para mostrar valores

print ("duracion_anios_negocio = ", duracion_anios_negocio)
print ("tasa_descuento = ", tasa_descuento)
print (type(tasa_descuento))

duracion_anios_negocio =  40
tasa_descuento =  0.1
<class 'float'>


In [ ]:
# 6
######################### Sección de Funciones #################################

# Función para convertir valores con ',' en puntos '.', extraer % etc
def convert_param_type(value):
  if isinstance(value, str):
    value = value.strip()
    if value.endswith('%'):
      try:
        # Eliminar el '%' y convertir a float, dividiendo por 100
        return round(float(value[:-1].replace(',', '.')) / 100, 2)
      except ValueError:
        return value # Dejar como texto si falla la conversión
    else:
      try:
        # Intentar convertir a numérico (int o float)
        if '.' in value or ',' in value:
            return float(value.replace(',', '.'))
        else:
            return int(value)
      except ValueError:
        return value # Dejar como texto si falla la conversión
  return value # Dejar el valor como está si no es string
################################################################################

# Función leer de la planilla de modelos, un rango específico y una hoja por defecto
# Función que lee la hoja "Parámetros Técnicos"
def leer_param_tecnicos(spreadsheet_name, sheet_range, sheet_name='Parámetros Técnicos'):
  """
  Reads the 'Parámetros Técnicos' sheet from a Google Spreadsheet,
  specifically the range B3:H14, and returns it as a pandas DataFrame.

  Args:
    spreadsheet_name: The name of the Google Spreadsheet.
    sheet_name: The name of the sheet to read (defaults to 'Parámetros Técnicos').

  Returns:
    A pandas DataFrame containing the data from the specified range.
  """
  spreadsheet = gc.open(spreadsheet_name)
  worksheet = spreadsheet.worksheet(sheet_name)

  # Get values from the specified range
  data = worksheet.get_values(sheet_range)

  header = data[0]
  data_rows = data[1:]

  df_param = pd.DataFrame(data_rows, columns=header)

  return df_param
################################################################################

# Función para obtener las variables del modelo
def get_variables_as_dict(spreadsheet_name, sheet_name='Variables del Modelo'):
  """
  Reads a specified sheet from a Google Spreadsheet and returns a dictionary
  where keys are variable names from the third column and values are the
  corresponding values from the second column.

  Args:
    spreadsheet_name: The name of the Google Spreadsheet.
    sheet_name: The name of the sheet to read.

  Returns:
    A dictionary containing variable names and their values.
  """
  spreadsheet = gc.open(spreadsheet_name)
  worksheet = spreadsheet.worksheet(sheet_name)

  # Get all the values from the worksheet as a list of lists
  data = worksheet.get_all_values()

  variables_dict = {}

  # Start from row 1 to skip headers
  for row in data[1:]:
    # Check if the row has at least 3 columns
    if len(row) >= 3:
      variable_name = row[2].strip() # Variable name is in the third column (index 2)
      cell_value = row[1].strip()   # Value is in the second column (index 1)
      variable_value = cell_value.replace(',', '.', 1) # Handle potential comma as decimal separator

      if variable_name: # Only process if variable name is not empty
        try:
          # Attempt to convert to a more specific type if possible
          if variable_value.lower() == 'true':
              variables_dict[variable_name] = True
          elif variable_value.lower() == 'false':
              variables_dict[variable_name] = False
          elif variable_value.isdigit():
              variables_dict[variable_name] = int(variable_value)
          else:
              try:
                  variables_dict[variable_name] = float(variable_value)
              except ValueError:
                  variables_dict[variable_name] = variable_value # Keep as string if conversion fails
          # print(f"Read variable: {variable_name} = {variables_dict[variable_name]}") # Optional: for debugging
        except Exception as e:
          print(f"Could not process row for variable '{variable_name}': {e}")
    # else:
      # print(f"Skipping row with insufficient columns: {row}") # Optional: for debugging

  return variables_dict
################################################################################

# Función para obtener la velocidad y el TN x eje de acuerdo al tipo de vía, trocha, etc
def get_velocidad_y_valor_trocha(tipo_via_inicio, vida_util_porc, tipo_trocha, vel_carga_dict):
  """
  Extrae la velocidad y el valor correspondiente al tipo_trocha desde
  el diccionario vel_carga_dict, basándose en el tipo de vía,
  el porcentaje de vida útil consumida y el tipo de trocha.

  Args:
    tipo_via_inicio: El tipo de vía de inicio del tramo (string).
    vida_util_porc: El porcentaje de vida útil consumida (float).
    tipo_trocha: El tipo de trocha del tramo (string).
    vel_carga_dict: El diccionario que contiene los datos de velocidad de carga.

  Returns:
    Una tupla (velocidad, valor_trocha), o (None, None) si no se encuentra
    una coincidencia.
  """
  for key, value in vel_carga_dict.items():
    tipo, desde, hasta = key
    # Asegurarse de que los valores 'desde' y 'hasta' son numéricos
    try:
      desde_num = float(desde)
      hasta_num = float(hasta)
    except ValueError:
      continue  # Saltar esta entrada si 'desde' o 'hasta' no son numéricos

    # Convertir el tipo_via del tramo actual a minúsculas para la comparación
    if (tipo_via_inicio == tipo) and (desde_num < vida_util_porc <= hasta_num):
      # La velocidad está en la columna 'velocidad'
      velocidad = value['velocidad']
      # El valor de la trocha está en la columna que coincide con tipo_trocha
      valor_trocha = value.get(tipo_trocha, None) # Usar .get para evitar KeyError si tipo_trocha no existe como columna

      return velocidad, valor_trocha

  return None, None
################################################################################

# Función que lee la planilla la sección de Costos y devuelve un dict con los datos
def get_costo_obra_dict ():
  df_param = leer_param_tecnicos('Copia de Modelo GF v9','a2:k14','Parámetros Técnicos')
  df_param_costo_obra = df_param.map(convert_param_type)
  costo_obra_dict = df_param_costo_obra.set_index(['tipo_inicial', 'tipo_obra']).to_dict(orient='index')
  return costo_obra_dict

################################################################################

def get_costo_obra (costo_obra_dict, tipo_via, obra, tipo_trocha):
  #get_costo_obra(tipo_via_inicio,'MEJOR1',tipo_trocha)
  return costo_obra_dict[(tipo_via, obra)][tipo_trocha]


################################################################################

# Función para obtener el costo de los desvíos
def get_costo_desvio(carga_util_proyectada, flag, incremento_cap_dict):
  """
  Busca en el diccionario incremento_cap_dict el rango de carga útil proyectada
  y devuelve el valor asociado a la clave 'long'.

  Args:
    incremento_cap_dict: Diccionario con información de incremento de capacidad,
                         indexado por un valor que representa un punto de corte.
                         Cada valor asociado a la clave debe ser un diccionario
                         con al menos las claves 'ton_anio' y 'long'.
                         Se espera que 'ton_anio' esté ordenado de forma ascendente
                         por la clave del diccionario principal.
    carga_util_proyectada: La carga útil proyectada (valor numérico) para buscar.

  Returns:
    El valor de 'long' asociado al rango donde cae la carga_util_proyectada.
    Retorna 0 si la carga útil es menor que el primer rango o si no se encuentra
    ningún rango coincidente.
  """
  # Ordenar las claves del diccionario para iterar en el orden correcto de 'ton_anio'

  if flag == 99:
    return 0,98

  sorted_keys = sorted(incremento_cap_dict.keys())

  # Manejar el caso donde la carga proyectada es menor que el primer umbral
  if carga_util_proyectada < incremento_cap_dict[sorted_keys[0]]['ton_anio']:
      return 0,98

  costo_desvio1 = valor_primera_renovacion * 1.2 * 1 if anio > 0 else 0

  # Iterar a través de los rangos definidos en el diccionario
  for i in range(len(sorted_keys) - 1):
      lower_bound = incremento_cap_dict[sorted_keys[i]]['ton_anio']
      upper_bound = incremento_cap_dict[sorted_keys[i+1]]['ton_anio']

      # Verificar si la carga proyectada cae en el rango actual
      # (lower_bound <= carga_util_proyectada < upper_bound)
      if carga_util_proyectada >= lower_bound and carga_util_proyectada < upper_bound:
          desvio = incremento_cap_dict[sorted_keys[i]]['long']
          flg=i
          #print("1", carga_util_proyectada, flag, desvio, flg)
          #return desvio * costo_desvio1, i

  # Si la carga proyectada es mayor o igual al último umbral,
  # devolver el 'long' asociado al último rango
  if carga_util_proyectada >= incremento_cap_dict[sorted_keys[-1]]['ton_anio']:
      desvio = incremento_cap_dict[sorted_keys[-1]]['long']
      flg=-1
      #print("2", carga_util_proyectada, flag, desvio, flg)
      #return desvio * costo_desvio1

  if flag != flg:
    desvio = desvio * costo_desvio1
    #print("3", carga_util_proyectada, flag, desvio, flg)
    return desvio, flg
  else:
    return 0, flag

  # Retornar 0 si no se encuentra un rango (debería cubrirse por los casos anteriores
  # si el diccionario está bien formado y cubre todos los casos, pero como fallback)

  return 0,98

################################################################################

# Exportación a un Google Sheet
# Dataframe, hoja de la planilla y la planilla
def export_gsheet(df_exp, nombre_hoja, celda, planilla="Resultados Modelo"):
  spreadsheet = gc.open(planilla)
  worksheet_salida = spreadsheet.worksheet(nombre_hoja)

  # Clear the existing content in the sheet
  RANGO_A_BORRAR = 'A2:U1500' # Tu rango como una cadena de texto
  worksheet_salida.batch_clear([RANGO_A_BORRAR])

  # Convert the DataFrame to a list of lists (including headers)
  #output_data = [df_exp.columns.values.tolist()] + df_exp.values.tolist()
  output_data = df_exp.values.tolist()

  # Update the Google Sheet with the data
  worksheet_salida.update(output_data, celda)
  #worksheet_salida.update(output_data,'A10')

  #print("DataFrame df_salida copied to 'Tramos' sheet.")

################################################################################


################## FUNCIONES DE SALIDA #########################################

def calc_suma_simple_por_anio(df, campo_carga):
    """
    Calcula la suma del campo de carga especificado para cada año hasta el año 40,
    sin dividir por la longitud del tramo. Si no hay datos para un año, pone 0.

    Args:
        df: pandas DataFrame con los resultados del modelo, debe contener
            las columnas 'año' y el campo especificado para la carga.
        campo_carga: El nombre de la columna que contiene la carga.

    Returns:
        Un pandas Series con la suma del campo de carga por año hasta el año 40,
        indexado por 'año'.
    """
    # Agrupar por 'año' y sumar el campo de carga
    suma_simple_por_anio = df.groupby('año')[campo_carga].sum()

    # Crear un índice completo hasta el año 40
    full_index = pd.RangeIndex(start=0, stop=41, step=1, name='año')

    # Reindexar la serie para incluir todos los años hasta el 40 y llenar los faltantes con 0
    suma_simple_por_anio = suma_simple_por_anio.reindex(full_index, fill_value=0)

    return suma_simple_por_anio

################################################################################

def calc_min_carga_por_anio(df, campo_carga):
    """
    Calcula la suma del campo de carga especificado para cada año hasta el año 40,
    sin dividir por la longitud del tramo. Si no hay datos para un año, pone 0.

    Args:
        df: pandas DataFrame con los resultados del modelo, debe contener
            las columnas 'año' y el campo especificado para la carga.
        campo_carga: El nombre de la columna que contiene la carga.

    Returns:
        Un pandas Series con la suma del campo de carga por año hasta el año 40,
        indexado por 'año'.
    """
    # Agrupar por 'año' y sumar el campo de carga
    min_por_anio = df.groupby('año')[campo_carga].min()

    # Crear un índice completo hasta el año 40
    full_index = pd.RangeIndex(start=1, stop=41, step=1, name='año')

    # Reindexar la serie para incluir todos los años hasta el 40 y llenar los faltantes con 0
    min_por_anio = min_por_anio.reindex(full_index, fill_value=0)

    return min_por_anio

################################################################################

def calc_suma_producto(df, campo_carga, anio, producto ):
  """
  Calcula la suma producto de la 'Carga útil teórica proyectada por año'
  multiplicado por la 'longitud' para cada 'id_tramo', considerando años > 0.

  Args:
    df: pandas DataFrame con los resultados del modelo, debe contener
        las columnas 'id_tramo', 'año', el campo especificado para la carga,
        y 'longitud'.
    campo_carga: El nombre de la columna que contiene la carga proyectada
                 (por defecto 'Carga útil teórica proyectada por año (q) (tn)').

  Returns:
    Un pandas Series con la suma producto para cada 'id_tramo',
    indexado por 'id_tramo'.
  """
  longitud_tramos = df_salida[df_salida['año'] == 0].groupby('año')['longitud'].sum().iloc[0]
  # print (longitud_tramos)

  # Crear un índice completo hasta el año 40
  full_index = pd.RangeIndex(start=0, stop=41, step=1, name='año')


  # Filtrar el DataFrame para incluir solo los años mayores a 0
  if anio==1:
    df_filtrado = df[df['año'] > 0].copy()
    full_index = pd.RangeIndex(start=1, stop=41, step=1, name='año')
  else:
    df_filtrado = df.copy()
    full_index = pd.RangeIndex(start=0, stop=41, step=1, name='año')

  # Calcular el producto de la carga y la longitud para los años filtrados
  if producto == 1:
    df_filtrado['carga_x_longitud'] = df_filtrado[campo_carga]*df_filtrado['longitud']
    #return df_filtrado[campo_carga]
  else:
    df_filtrado['carga_x_longitud'] = (df_filtrado[campo_carga] * df_filtrado['longitud'])/longitud_tramos

  # Calcular la suma de 'carga_x_longitud' agrupando por 'id_tramo'

  suma_producto_por_anio = df_filtrado.groupby('año')['carga_x_longitud'].sum()

  # Reindexar la serie para incluir todos los años hasta el 40 y llenar los faltantes con 0
  suma_producto_por_anio = suma_producto_por_anio.reindex(full_index, fill_value=0)

  return suma_producto_por_anio

################################################################################

def get_carga_util_proyectada(carga_util_teorica, tope_capacidad_transporte):

    if carga_util_teorica > tope_capacidad_transporte:
      return tope_capacidad_transporte
    else:
      return carga_util_teorica

################################################################################


In [ ]:
# 7

########################### Leo loss Datos del Tramo ###########################

# esta en la hoja "Listado de Tramos"
# tramos = leer_listado_tramos()
# devuelve un DF

df_tramos = leer_listado_tramos('Copia de Modelo GF v9', "Listado de Tramos")

## Datos Tramos (variables)
# id_tramo
# tipo_trocha
# tipo_via_inicio
# consumo_vida_util_actual = .7 ## un 100 % quiere decir que se consumió toda la vida util
# carga_util_teorica = 5000 # Cap de transporte en miles de TN
# long_tramo = 2
# tipo_via_final
# barrera

df_tramos['tipo_trocha'] = df_tramos['tipo_trocha'].str.lower()
# Esta división la uso para tener todo en valores de miles -- ya cambié todo en la planilla
df_tramos['carga_util_teorica'] = pd.to_numeric(df_tramos['carga_util_teorica'], errors='coerce')/1000
#df_tramos['carga_util_teorica'] = df_tramos['carga_util_teorica'].astype(int)

df_tramos

,id_tramo,long_tramo,linea,operador,division,desde_km,hasta_km,tipo_serv,tipo_via_inicio,consumo_vida_util_actual,veloc_inicial,carga_util_teorica,tipo_trocha,tipo_via_final,barreras
0,344,1.06,NaN,NaN,NaN,1086.648,1088.000,NaN,II,0.85,22,144.020,angosta,,NaN
1,346,1.63,NaN,NaN,NaN,1092.000,1093.621,NaN,II,0.80,40,144.020,angosta,,NaN
2,345,3.91,NaN,NaN,NaN,1088.000,1092.000,NaN,II,0.80,50,144.020,angosta,,NaN
3,348,4.35,NaN,NaN,NaN,1098.890,1103.204,NaN,II,0.80,50,144.020,angosta,,NaN
4,350,2.85,NaN,NaN,NaN,1108.130,1111.000,NaN,II,0.80,50,144.020,angosta,,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2001,2284,1.70,NaN,NaN,5.0,1202.000,1203.700,NaN,III,0.65,40,24.416,ancha,,NaN
2002,2802,1.01,NaN,NaN,35.0,892.000,893.000,NaN,II,0.81,20,1786.521,ancha,,NaN
2003,2801,13.99,NaN,NaN,35.0,878.224,892.000,NaN,II,0.56,50,1786.521,ancha,,NaN
2004,2266,12.50,NaN,NaN,5.0,1108.000,1120.500,NaN,III,0.86,50,488.545,ancha,,NaN


In [ ]:
#print(df_tramos.info())

print(df_tramos.describe())


          id_tramo   long_tramo  linea  operador    division     desde_km  \
count  2006.000000  2006.000000    0.0       0.0  831.000000  2006.000000   
mean   1965.086241     6.763973    NaN       NaN   23.139591   457.667415   
std    1210.169230     6.118480    NaN       NaN   23.984327   400.201767   
min     125.000000     0.500000    NaN       NaN    1.000000     0.000000   
25%     876.250000     2.000000    NaN       NaN    4.000000   101.598750   
50%    1873.500000     4.555000    NaN       NaN    8.000000   355.476500   
75%    2838.500000    10.167500    NaN       NaN   40.000000   723.025000   
max    4623.000000    41.870000    NaN       NaN   79.000000  1702.500000   

          hasta_km  tipo_serv  consumo_vida_util_actual  veloc_inicial  \
count  2006.000000        0.0               2006.000000    2006.000000   
mean    464.400872        NaN                  0.606535      38.032901   
std     400.808129        NaN                  0.275803      15.489516   
min       

In [ ]:
# 8
# Hay una función que lee directo de excel a dict
########################### Convierto en DF ####################################

## Leo los parámetros técnicos de la planilla y cargo varios df
## luego a esos df hay que convertirlos en dict

#df_param = leer_param_tecnicos('Copia de Modelo GF v9','a2:k14','Parámetros Técnicos')
#df_param_costo_obra = df_param.map(convert_param_type)

#print(df_param_costo_obra)
#print(df_param_costo_obra .info())
#print()

df_param = leer_param_tecnicos('Copia de Modelo GF v9','a17:c21','Parámetros Técnicos')
df_param_vida_util = df_param.map(convert_param_type)

print(df_param_vida_util)
print(df_param_vida_util.info())
print()

df_param = leer_param_tecnicos('Copia de Modelo GF v9','a24:d28','Parámetros Técnicos')
df_param_incremento_cap = df_param.map(convert_param_type)

print(df_param_incremento_cap)
print(df_param_incremento_cap.info())
print()

df_param = leer_param_tecnicos('Copia de Modelo GF v9','a31:h47','Parámetros Técnicos')
df_param_vel_carga = df_param.map(convert_param_type)

print(df_param_vel_carga)
print(df_param_vel_carga.info())
print()

df_param = leer_param_tecnicos('Copia de Modelo GF v9','a50:h55','Parámetros Técnicos')
df_param_costo_mantenimiento = df_param.map(convert_param_type)

print(df_param_costo_mantenimiento)
print(df_param_costo_mantenimiento.info())

df_param = leer_param_tecnicos('Copia de Modelo GF v9','a63:c66','Parámetros Técnicos')
df_param_pasosanivel = df_param.map(convert_param_type)

print(df_param_pasosanivel)
print(df_param_pasosanivel.info())

           item  vida_util tipo
0    Vía Tipo I     300000    I
1   Vía Tipo II     220000   II
2  Vía Tipo III     100000  III
3   Vía Tipo IV     100000   IV
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   item       4 non-null      object
 1   vida_util  4 non-null      int64 
 2   tipo       4 non-null      object
dtypes: int64(1), object(2)
memory usage: 228.0+ bytes
None

                     item  ton_anio  long  indice
0   Carga anual > 4000000      4000  0.05       1
1   Carga anual > 6000000      6000  0.10       2
2   Carga anual > 8000000      8000  0.20       3
3  Carga anual > 12000000     12000  1.00       4
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   item      4 non-null      object 
 1   ton_anio  4 

In [ ]:
# 9

##################### Covierto en DICT #########################################
# Acá se convierten los df en dict

# 1. OK - Convertir df_param_costo_obra a diccionario con 'tipo_inicial' como clave
#costo_obra_dict = df_param_costo_obra.set_index(['tipo_inicial', 'tipo_obra']).to_dict(orient='index')
#print(costo_obra_dict)
#print()

# 2. OK - Convertir df_param_vida_util a diccionario con 'tipo' como clave
vida_util_dict = df_param_vida_util.set_index('tipo').to_dict(orient='index')
# print("Diccionario de df_param_vida_util con 'tipo' como clave:")
print(vida_util_dict)
print()

# 3. OK - Convertir df_param_incremento_cap a diccionario con 'tipo' como clave
incremento_cap_dict = df_param_incremento_cap.set_index('indice').to_dict(orient='index')
# print("Diccionario de df_param_incremento_cap con 'tipo' como clave:")
print(incremento_cap_dict)
print()

# 4. OK - Convertir df_param_vel_carga a diccionario con 'tipo_inicial' como clave
vel_carga_dict = df_param_vel_carga.set_index(['tipo','desde','hasta']).to_dict(orient='index')
# print("Diccionario de df_param_vel_carga con 'tipo_inicial' como clave:")
print(vel_carga_dict)
print()

# 5. OK - Convertir df_param_costo_mantenimiento a diccionario con 'tipo_inicial' como clave
costo_mantenimiento_dict = df_param_costo_mantenimiento.set_index('tipo').to_dict(orient='index')
# print("Diccionario de df_param_costo_mantenimiento con 'tipo_inicial' como clave:")
print(costo_mantenimiento_dict)
print()

# 6. OK - Convertir df_param_costo_mantenimiento a diccionario con 'tipo_inicial' como clave
pasosanivel_dict = df_param_pasosanivel.set_index('pasos a nivel').to_dict(orient='index')
# print("Diccionario de df_param_costo_mantenimiento con 'tipo_inicial' como clave:")
# pan_carga_ref1 = df_param_pasosanivel.iloc[0, 1]
# pan_carga_ref2 = df_param_pasosanivel.iloc[1, 1]
# pan_cubi1 = df_param_pasosanivel.iloc[0, 2]
# pan_cubi2 = df_param_pasosanivel.iloc[1, 2]
# pan_cubi3 = df_param_pasosanivel.iloc[2, 2]

print (pasosanivel_dict)
#print(pan_carga_ref1, pan_carga_ref2, pan_cubi1, pan_cubi2, pan_cubi3)
print()

{'I': {'item': 'Vía Tipo I', 'vida_util': 300000}, 'II': {'item': 'Vía Tipo II', 'vida_util': 220000}, 'III': {'item': 'Vía Tipo III', 'vida_util': 100000}, 'IV': {'item': 'Vía Tipo IV', 'vida_util': 100000}}

{1: {'item': 'Carga anual > 4000000', 'ton_anio': 4000, 'long': 0.05}, 2: {'item': 'Carga anual > 6000000', 'ton_anio': 6000, 'long': 0.1}, 3: {'item': 'Carga anual > 8000000', 'ton_anio': 8000, 'long': 0.2}, 4: {'item': 'Carga anual > 12000000', 'ton_anio': 12000, 'long': 1.0}}

{('I', 0.0, 0.8): {'item': 'Hasta 80% - Vía Tipo I', 'velocidad': 70, 'ancha': 25, 'media': 23, 'angosta': 22}, ('II', 0.0, 0.8): {'item': 'Hasta 80% - Vía Tipo II', 'velocidad': 60, 'ancha': 22, 'media': 21, 'angosta': 20}, ('III', 0.0, 0.8): {'item': 'Hasta 80% - Vía Tipo III', 'velocidad': 55, 'ancha': 20, 'media': 19, 'angosta': 18}, ('IV', 0.0, 0.8): {'item': 'Hasta 80% - Vía Tipo IV', 'velocidad': 55, 'ancha': 20, 'media': 19, 'angosta': 18}, ('I', 0.8, 0.95): {'item': 'Entre 80% y 95% - Vía Tipo I

In [ ]:
# 10

##################### Cargo las Variables tomadas desde los dict ###############

def load_variables_dict():
  global tope_capacidad_transporte, vida_util_renovada, vida_util_actual, vida_util_mejorada, vida_util_resto, vida_util_porc
  global valor_primera_renovacion, valor_primer_mejoramiento1, anios_obra_renovacion, anios_obra_mejoramiento1, rejuve_obra_mejoramiento1
  global consumo_vida_util, consumo_vida_util_actual, disparo_obra_renovacion, disparo_obra_mejoramiento1, disparo_obra_mejoramiento2
  global duracion_anios_negocio, tasa_descuento, crec_factor_inicial, periodo_anios_crec_inicial, crec_factor_final, canon_comercial
  global costo_desvio, costo_fijo_conservacion, costo_manten_inicial, costo_manten1, costo_manten_renov, carga_ref_manten, tope_inicial_manten
  global crec_manten, mantenimiento_renovacion, carga_ref_renov, tope_inicial_renov, crec_renov, repe_obra_renov, repe_obra_mejor1, repe_obra_mejor2
  global tipo_via_inicio, tipo_trocha, crec_factor_inicial, crec_factor_final, disparo_obra_mejoramiento1
  global cubi_pan1, cubi_pan2, cubi_pan3, carga_ref_pan1, carga_ref_pan2, repe_obra_mejor2, costo_obra_dict
  global valor_primer_mejoramiento2, anios_obra_mejoramiento2, rejuve_obra_renov, marca_mejoramiento1, marca_mejoramiento2, marca_renovacion
  global rejuve_obra_mejoramiento1, rejuve_obra_mejoramiento2, disparo_obra_mejoramiento2

  costo_obra_dict = get_costo_obra_dict()
  #print (costo_obra_dict)

  # Valor máximo de capacidad de transporte
  #tope_capacidad_transporte = incremento_cap_dict[]
  tope_capacidad_transporte = incremento_cap_dict[max(incremento_cap_dict.keys())]['ton_anio']
  #print (tope_capacidad_transporte)

  # Valor máximo de vida útil cuando la vía se renueva
  # Todo se renueva a tipo_via = I y 3000 millones
  vida_util_renovada = vida_util_dict['I']['vida_util']
  #print(vida_util_renovada)

  # Valor vida útil cuando empezamos el loop
  vida_util_actual = vida_util_dict[tipo_via_inicio]['vida_util']
  #print(vida_util_actual)

  # Valor obra, años y disparo renovación
  #print ("Costo, años, disparo, rejuvenecimiento")
  valor_primera_renovacion = costo_obra_dict[tipo_via_inicio,'RENOV'][tipo_trocha]
  anios_obra_renovacion = costo_obra_dict[tipo_via_inicio,'RENOV']['plazo_int']
  disparo_obra_renovacion = costo_obra_dict[tipo_via_inicio,'RENOV']['momento_vida_util']
  rejuve_obra_renovacion = costo_obra_dict[tipo_via_inicio,'RENOV']['rejuvenecimiento']
  repe_obra_renov = costo_obra_dict[tipo_via_inicio,'RENOV']['limite_intervencion']

  costo_desvio = valor_primera_renovacion * 1.2
  #print(valor_primera_renovacion, anios_obra_renovacion, disparo_obra_renovacion, rejuve_obra_renovacion, costo_desvio)

  # Valor obra, años y disparo mejoramiento 1
  valor_primer_mejoramiento1 = costo_obra_dict[tipo_via_inicio,'MEJOR1'][tipo_trocha]
  anios_obra_mejoramiento1 = costo_obra_dict[tipo_via_inicio,'MEJOR1']['plazo_int']
  disparo_obra_mejoramiento1 = costo_obra_dict[tipo_via_inicio,'MEJOR1']['momento_vida_util']
  rejuve_obra_mejoramiento1 = costo_obra_dict[tipo_via_inicio,'MEJOR1']['rejuvenecimiento']
  repe_obra_mejor1 = costo_obra_dict[tipo_via_inicio,'MEJOR1']['limite_intervencion']
  #print(valor_primer_mejoramiento1, anios_obra_mejoramiento1, disparo_obra_mejoramiento1, rejuve_obra_mejoramiento1)

  # Valor obra, años y disparo mejoramiento 2
  valor_primer_mejoramiento2 = costo_obra_dict[tipo_via_inicio,'MEJOR2'][tipo_trocha]
  anios_obra_mejoramiento2 = costo_obra_dict[tipo_via_inicio,'MEJOR2']['plazo_int']
  disparo_obra_mejoramiento2 = costo_obra_dict[tipo_via_inicio,'MEJOR2']['momento_vida_util']
  rejuve_obra_mejoramiento2 = costo_obra_dict[tipo_via_inicio,'MEJOR2']['rejuvenecimiento']
  repe_obra_mejor2 = costo_obra_dict[tipo_via_inicio,'MEJOR2']['limite_intervencion']
  #print(valor_primer_mejoramiento2, anios_obra_mejoramiento2, disparo_obra_mejoramiento2, rejuve_obra_mejoramiento2)

  vida_util_porc=consumo_vida_util_actual
  consumo_vida_util=vida_util_dict[tipo_via_inicio]['vida_util']
  vida_util_mejorada = vida_util_dict[tipo_via_inicio]['vida_util']

  #print (vida_util_porc, consumo_vida_util)

  costo_fijo_conservacion = costo_mantenimiento_dict[0][tipo_trocha]
  costo_manten_inicial = costo_mantenimiento_dict[tipo_via_inicio][tipo_trocha]
  carga_ref_manten = costo_mantenimiento_dict[tipo_via_inicio]['carga_ref']
  tope_inicial_manten = costo_mantenimiento_dict[tipo_via_inicio]['tope']
  crec_manten = costo_mantenimiento_dict[tipo_via_inicio]['crec']

  mantenimiento_renovacion = costo_mantenimiento_dict['I'][tipo_trocha]
  carga_ref_renov = costo_mantenimiento_dict['I']['carga_ref']
  tope_inicial_renov = costo_mantenimiento_dict['I']['tope']
  crec_renov = costo_mantenimiento_dict['I']['crec']

  # print (costo_fijo_conservacion)
  # print (costo_manten_inicial, carga_ref_manten, tope_inicial_manten, crec_manten)
  # print (mantenimiento_renovacion, carga_ref_renov, tope_inicial_renov, crec_renov)

  #print (mantenimiento_inicial, carga_ref_manten, tope_inicial_manten, crec_manten

  carga_ref_pan1 = pasosanivel_dict['Carga de Referencia <Ref 1']['carga_referencia']
  carga_ref_pan2 = pasosanivel_dict['Carga de Referencia entre']['carga_referencia']
  cubi_pan1 = pasosanivel_dict['Carga de Referencia <Ref 1']['cubi']
  cubi_pan2 = pasosanivel_dict['Carga de Referencia entre']['cubi']
  cubi_pan3 = pasosanivel_dict['Carga de Referencia >Ref 1']['cubi']

  #print (carga_ref_pan1, carga_ref_pan2, cubi_pan1, cubi_pan2, cubi_pan3)

##### VER QUE CUANDO HAY UNA RENOV SE ALTERA EL VALOR DE LOS MEJORAMIENTOS tipo_via
#####



In [ ]:
# 11

#################################### MAIN ######################################

#################################### leo los tramos ############################

# partiendo del df de tramos, recorrer todas las filas extrayendo los siguientes campos "consumo_vida_util_actual", "tipo_via_inicio", "carga_util_teorica" y "tipo_trocha". Cargar esos valores en variables e invocar una función que todavia no esta definida

import time

df_tramos = leer_listado_tramos('Copia de Modelo GF v9', "Listado de Tramos")
df_tramos['tipo_trocha'] = df_tramos['tipo_trocha'].str.lower()
# # Esta división la uso para tener todo en valores de miles -- ya cambié todo en la planilla
# #df_tramos['carga_util_teorica'] = pd.to_numeric(df_tramos['carga_util_teorica'], errors='coerce')/1000
df_tramos['carga_util_teorica'] = df_tramos['carga_util_teorica'].astype(int)

# leer_var_modelo('Copia de Modelo GF v9', 'Variables del Modelo')

start_time = time.time()

salida = []
total_registros = len(df_tramos)

# Recorrer las filas del DataFrame df_tramos
for index, row in df_tramos.iterrows():
# Extraer los campos a variables
  id_tramo = row['id_tramo']
  consumo_vida_util_actual = row['consumo_vida_util_actual']
  tipo_via_inicio = row['tipo_via_inicio']
  carga_util_teorica = row['carga_util_teorica']/1000
  tipo_trocha = row['tipo_trocha']
  barreras = row['barreras']
  long_tramo = row['long_tramo']
  linea = row['linea']
  operador = row['operador']
  division = row['division']
  desde_km = row['desde_km']
  hasta_km = row['hasta_km']
  tipo_serv =  row['tipo_serv']


  load_variables_dict() ## Carga las variables desde los DF o Dict

  ### --------------------------------------------------------------------------

  # 1ro invocar las variables del modelo
  # leer las hojas y armar los dict
  # entrar en un loop para leer los datos del tramo
  # main_calc debería recibir los datos del tramo y los dict de datos
  # dentro del main debería realizar los calculos
  # y devolver un df con todo calculado

  ### Función que imprime los años
  ###
  ### COMIENZO LOOP AÑOS
  ###

  ## Cargo los datos del tramo activo
  # id_tramo = row['id_tramo']
  # consumo_vida_util_actual = row['consumo_vida_util_actual']
  # tipo_via_inicio = row['tipo_via_inicio']
  # carga_util_teorica = row['carga_util_teorica']
  # tipo_trocha = row['tipo_trocha']
  # barreras = row['barreras']
  # long_tramo = row['long_tramo']

  #load_variables_dict()

  ##################### Mantenimiento
  # mantenimiento_renov = 6
  # carga_ref_renov = 3000 # 3000
  # tope_inicial_renov = 70
  # crec_renov = 0.03

  # mantenimiento_inicial = 4.5
  # carga_ref_manten = 1000 # 1000
  # tope_inicial_manten = 50
  # crec_manten = 0.03

  costo_manten1 =0
  costo_manten_renov = 0
  costo_fijo = 0

  #PCT falta

  #####################

    # 1ro invocar las variables del modelo
  # leer las hojas y armar los dict
  # entrar en un loop para leer los datos del tramo
  # main_calc debería recibir los datos del tramo y los dict de datos
  # dentro del main debería realizar los calculos
  # y devolver un df con todo calculado

  ### Función que imprime los años
  ###
  ### COMIENZO LOOP AÑOS
  ###

  ## Cargo los datos del tramo activo
  # id_tramo = row['id_tramo']
  # consumo_vida_util_actual = row['consumo_vida_util_actual']
  # tipo_via_inicio = row['tipo_via_inicio']
  # carga_util_teorica = row['carga_util_teorica']
  # tipo_trocha = row['tipo_trocha']
  # barreras = row['barreras']
  # long_tramo = row['long_tramo']


  ##################### Mantenimiento
  # mantenimiento_renov = 6
  # carga_ref_renov = 3000 # 3000
  # tope_inicial_renov = 70
  # crec_renov = 0.03

  # mantenimiento_inicial = 4.5
  # carga_ref_manten = 1000 # 1000
  # tope_inicial_manten = 50
  # crec_manten = 0.03

  costo_manten1 =0
  costo_manten_renov = 0
  costo_fijo = 0

  #PCT falta

  #####################

  #salida = []

  ###################### Inicialización Variables aux Obra ##
  obra = 0 ## Indica si hay una obra en ejecución
  anios_obra = 0
  anios_obra_renov = 0
  anios_obra_mejor1 = 0
  anios_obra_mejor2 = 0

  valor_obra = 0
  anios_obra_rm = 1

  #repe_obra_renov = 1
  #repe_obra_mejor1 = 1
  #repe_obra_mejor2 = 1

  marca_mejoramiento1 = ""
  marca_mejoramiento2 = ""
  marca_renovacion = ""
  marca = ""

  cant_obra_mejor1 = 0
  cant_obra_mejor2 = 0
  cant_obra_renov = 0

  vida_util = vida_util_actual*consumo_vida_util_actual

  repe_obra_renov   = repe_obra_renov - 1
  repe_obra_mejor1  = repe_obra_mejor1 - 1
  repe_obra_mejor2  = repe_obra_mejor2 - 1

  vida_util_resto = vida_util_actual * (1 - consumo_vida_util_actual)
  ###################### Fin variables Obra

  costo_desvio_aplicado = 0
  costo_desvio = 0

  # flg1  =True
  # flg2  =True
  # flg3  =True
  # flg4  =True
  # anio = 0

  acum_obra = 0
  costo_oper_infra_tramo = 0
  veloc = 0
  carga_eje = 0

  flag_des = 99

  duracion_anios_negocio = 40

  print("Año Vía Tipo   Carga Util  Carga Util  TNKM      Consumo     Renov   Costo   Costo  Cos 1er Cos 2do  Costo    Veloc   TN x Eje")
  print("        Trocha  Teórica     Proyectada            Vida Util   y Mejor Desvio  Fijo   Manten  Manten  Operación")
  print("               (TN x mil)  (TN x mil) (x mil)  (x mil)  (%)                                          Infra")

  for anio in range (0, duracion_anios_negocio + 1):
    ## Acá se estipulas dos tipos de crecimiento uno inicial con un % y posteriormente otro
    if anio < periodo_anios_crec_inicial:
      crecimiento = crec_factor_inicial
    else:
      crecimiento = crec_factor_final

    carga_util_proyectada = get_carga_util_proyectada(carga_util_teorica, tope_capacidad_transporte)

    desvio = get_costo_desvio(carga_util_proyectada, flag_des, incremento_cap_dict)
    costo_desvio_aplicado = desvio[0]
    flag_des = desvio[1]

    ###
    ### SALIDA IMPRESION -> Futuro DF
    ###
    ###'''
    print(f"{anio:<5}"
          f"{tipo_via_inicio:<4}"
          f"{tipo_trocha:<9}"
          f"{int(carga_util_teorica):<12} "
          f"{int(carga_util_proyectada):<9}"
          #"{:.2f}".format(carga_util_proyectada*long_tramo),
          f"{int(carga_util_proyectada*long_tramo):<9}"
          f"{int(vida_util):<9}"
          f"{vida_util_porc:<8.0%}"
          ##f"{vida_util_resto:<8.0f}"
          f"{valor_obra/anios_obra_rm:<6.0f}"
          ##f"{valor_mejor1/anios_obra_mejoramiento1:<6.0f}"
          f"{costo_desvio_aplicado:<7.0f}"
          f"{costo_fijo:<7.0f}"
          f"{costo_manten1:<9.0f}"
          f"{costo_manten_renov:<9.0f}"
          f"{costo_oper_infra_tramo:<9.0f}"
          f"{veloc:<9.0f}"
          f"{carga_eje:<9.0f}"
          f"{obra:<7}"
          f"{marca}"
          )
    ### '''
    # Sirve para diferenciar entre obras de renovación y mejoramiento
    if obra == 1:
      valor_obra_rn = valor_obra
      anios_obra_rn = anios_obra_rm
      valor_obra_rj = 0
    elif obra == 2 or obra == 3:
      valor_obra_mj = valor_obra
      anios_obra_mj = anios_obra_rm
      valor_obra_rn = 0
    else:
      valor_obra_rn = 0
      valor_obra_mj = 0
      anios_obra_rn = 1
      anios_obra_mj = 1

    salida.append({
        'id_tramo': id_tramo,
        'longitud': long_tramo,
        'tipo_trocha': tipo_trocha,
        'tipo_via_inicio': tipo_via_inicio,
        'año': anio,
        'Carga útil teórica proyectada por año (q) (tn)': int(carga_util_teorica),
        'Carga proyectada por año (q) (tn) - LIMITADA': int(carga_util_proyectada),
        'Carga anual TN.KM': round(float(carga_util_proyectada*long_tramo),2),
        'Carga equiv. desde última intervención': int(vida_util),
        '% Vida útil consumida': round(vida_util_porc,2),
        'Obra de Renovación': round(float(valor_obra_rn/anios_obra_rn),2),
        'Obra de Mejoramiento': round(float(valor_obra_mj/anios_obra_mj),2),
        'Desvíos a construir': round(float(costo_desvio_aplicado),2),
        'Conservación costo fijo anual': round(float(costo_fijo),2),
        'Primer Tramo de Mantenimiento': round(float(costo_manten1),2),
        'Segundo Tramo de Mantenimiento': round(float(costo_manten_renov),2),
        'Costo operación infraestructura del tramo': round(float(costo_oper_infra_tramo),2),
        'Velocidad': int(veloc),
        'Carga x Eje':int(carga_eje),
        'tipo_obra': int(obra),
        'marca': marca
    })

    carga_util_teorica = carga_util_teorica * (1+crecimiento)

    vida_util = vida_util+get_carga_util_proyectada(carga_util_teorica, tope_capacidad_transporte) # Carga equiv. desde última intervención

    #carga_util_proyectada * (1+crecimiento)

    vida_util_porc = vida_util/consumo_vida_util

    vida_util_resto = vida_util_resto-get_carga_util_proyectada(carga_util_teorica, tope_capacidad_transporte)

    #print(vida_util_resto

    if (marca == "Comienzo Obra Mejoramiento" or marca == "Comienzo Obra Renovación"):
      marca = ""

    if (marca == "Obra Finalizada" or marca == "Obra Renovación Finalizada"):
      anios_obra_rm=1
      valor_obra = 0
      #obra = 0
      marca = ""

    #print(vida_util_porc, disparo_obra_renovacion, cant_obra_renov, repe_obra_renov, obra)


      ## Se dispara obra renovación
    if (vida_util_porc >= disparo_obra_renovacion and cant_obra_renov<=repe_obra_renov and obra == 0): # or obra == 1)):
      anios_obra_renov += 1
      #if anios_obra_renov == 1:
      marca = "Comienzo Obra Renovación"
      obra = 1
      valor_obra = valor_primera_renovacion
      anios_obra_rm = anios_obra_renovacion
      acum_obra = valor_primera_renovacion/anios_obra_renovacion
    elif (obra == 1 and anios_obra_renov >= anios_obra_renovacion):  ## Si hay obra en ejecución del tipo renovación y los años de esa obra se alcanzaron se renueva la vida útil ## RENOVACION
      vida_util = get_carga_util_proyectada(carga_util_teorica, tope_capacidad_transporte)# * (1+crecimiento)
      vida_util_porc = vida_util/consumo_vida_util
      vida_util_resto = vida_util_renovada - get_carga_util_proyectada(carga_util_teorica, tope_capacidad_transporte) #carga_util_proyectada * (1+crecimiento)
      anios_obra_renov = 0
      cant_obra_renov += 1
      valor_obra = 0
      obra = 0
      marca = "Obra Renovación Finalizada"
      tipo_via_inicio = 'I'
      consumo_vida_util = vida_util_renovada
      vida_util_porc = vida_util/consumo_vida_util
      valor_primer_mejoramiento1 = get_costo_obra(costo_obra_dict,tipo_via_inicio,'MEJOR1',tipo_trocha)
      valor_primer_mejoramiento2 = get_costo_obra(costo_obra_dict,tipo_via_inicio,'MEJOR2',tipo_trocha)
    elif (obra == 1 and anios_obra_renov < anios_obra_renovacion):
      marca = "... Sigue obra Renovación"
      anios_obra_renov += 1
      acum_obra = (valor_primera_renovacion/anios_obra_renovacion)*anios_obra_renov

    ## Se dispara obra mejoramiento 1
    if (vida_util_porc >= disparo_obra_mejoramiento1 and cant_obra_mejor1<=repe_obra_mejor1 and obra == 0): # or obra == 2)):
      #anios_obra_mejor1 += 1
      anios_obra_mejor1 = 1
      #if anios_obra_mejor1 == 1:
      marca = "Comienzo Obra Mejoramiento 1"
      obra = 2
      valor_obra = valor_primer_mejoramiento1
      anios_obra_rm = anios_obra_mejoramiento1
    elif (obra == 2 and anios_obra_mejor1 >= anios_obra_mejoramiento1): ## MEJORAMIENTO 1 ## Si hay obra en ejecución y los años de esa obra se alcanzaron se actualiza el valor la vida útil
      vida_util = (vida_util-get_carga_util_proyectada(carga_util_teorica, tope_capacidad_transporte))*(1-rejuve_obra_mejoramiento1)+get_carga_util_proyectada(carga_util_teorica,tope_capacidad_transporte)
      vida_util_porc = vida_util/consumo_vida_util
      vida_util_resto = vida_util_mejorada - vida_util
      ##resto * (1+porc_valor_primer_mejoramiento1)
      obra = 0
      anios_obra_mejor1 = 0
      cant_obra_mejor1 += 1
      valor_obra = 0
      marca = "Obra Finalizada"
      ## Tipo de Vía se deja igual
    elif (obra == 2 and anios_obra_mejor1 < anios_obra_mejoramiento1):
      anios_obra_mejor1 += 1
      marca ="... Sigue obra Mejoramiento 1"

  ## Se dispara obra mejoramiento 2
    if (vida_util_porc >= disparo_obra_mejoramiento2 and cant_obra_mejor2<=repe_obra_mejor2 and obra == 0): # or obra == 2)):
      #anios_obra_mejor1 += 1
      anios_obra_mejor2 = 1
      #if anios_obra_mejor1 == 1:
      marca = "Comienzo Obra Mejoramiento 2"
      obra = 3
      valor_obra = valor_primer_mejoramiento2
      anios_obra_rm = anios_obra_mejoramiento2
    elif (obra == 3 and anios_obra_mejor2 >= anios_obra_mejoramiento2): ## MEJORAMIENTO 2 ## Si hay obra en ejecución y los años de esa obra se alcanzaron se actualiza el valor la vida útil
      vida_util = (vida_util-carga_util_proyectada)*(1-rejuve_obra_mejoramiento1)+carga_util_proyectada
      vida_util_porc = vida_util/consumo_vida_util
      vida_util_resto = vida_util_mejorada - vida_util
      ##resto * (1+porc_valor_primer_mejoramiento1)
      obra = 0
      anios_obra_mejor2 = 0
      cant_obra_mejor2 += 1
      valor_obra = 0
      marca = "Obra Finalizada"
      ## Tipo de Vía se deja igual
    elif (obra == 3 and anios_obra_mejor2 < anios_obra_mejoramiento2):
      anios_obra_mejor2 += 1
      marca ="... Sigue obra Mejoramiento 2"


  # Costo mantenimiento 1

    costo_manten1 = costo_manten_inicial *((1+crec_manten)**(vida_util/carga_ref_manten))

    if costo_manten1 >= tope_inicial_manten:
      costo_manten1 = tope_inicial_manten

    if acum_obra > 0:
      costo_manten1 = costo_manten1 *((valor_primera_renovacion-acum_obra)/valor_primera_renovacion)

  # Costo mantenimiento 2 o costo aplicado a una vía renovada
    if anios_obra_renov == 0 and acum_obra == 0:
        costo_manten_renov = 0
    elif anios_obra_renov>0 and acum_obra > 0:
        factor_renov = (acum_obra)/valor_primera_renovacion
        costo_manten_renov = mantenimiento_renovacion * ((1+crec_renov)**(carga_util_proyectada/carga_ref_renov)) * factor_renov
        #costo_manten_renov = mantenimiento_renov * ((1+crec_renov)**(vida_util/carga_ref_renov)) * factor_renov
        #costo_manten_renov=98
    elif anios_obra_renov==0 and acum_obra > 0:
        factor_renov = (acum_obra)/valor_primera_renovacion
        #costo_manten_renov = mantenimiento_renov * ((1+crec_renov)**(carga_util_proyectada/carga_ref_renov)) * factor_renov
        costo_manten_renov = mantenimiento_renovacion * ((1+crec_renov)**(vida_util/carga_ref_renov)) * factor_renov
        #costo_manten_renov=97

    if costo_manten_renov> tope_inicial_renov:
      costo_manten_renov = tope_inicial_renov

  #  print (carga_ref_pan1, carga_ref_pan2, cubi_pan1, cubi_pan2, cubi_pan3)
    if carga_util_proyectada < carga_ref_pan1:
      costo_infra_tramo = cubi_pan1
    elif carga_util_proyectada >= carga_ref_pan1 and carga_util_proyectada < carga_ref_pan2:
      costo_infra_tramo = cubi_pan2
    else:
      costo_infra_tramo = cubi_pan3

########################## LONGITUD y BARRERAS #################################
    if long_tramo < 1:
      long_tramo = 1

    if barreras == 0 or pd.isna(barreras):
      barreras = 1
########################## LONGITUD y BARRERAS #################################

    costo_oper_infra_tramo = barreras * (costo_infra_tramo/long_tramo)

    costo_fijo = costo_fijo_conservacion

    veltn = get_velocidad_y_valor_trocha(tipo_via_inicio, vida_util_porc, tipo_trocha, vel_carga_dict)
    veloc = veltn[0]
    carga_eje = veltn[1]

    if (anio ==60):
      #df_salida = pd.DataFrame(salida)
      # sirve para detener la corrida en algún año específico
      break

  #print()
  # Calcular el porcentaje de progreso
  porcentaje_progreso = ((index + 1) / total_registros) * 100

  # Imprimir el progreso (puedes ajustar la frecuencia de impresión)

  #if (index + 1) % 5 == 0 or (index + 1) == total_registros: # Imprime cada 5 registros o al final
  print(f"Procesando registro {index + 1}/{total_registros} - Progreso: {porcentaje_progreso:.2f}%")

  ## Sirve para detener la corrida en el primer tramo

  #break


# Calculate the elapsed time
elapsed_time = time.time() - start_time
print(f"\nEl código tardó {elapsed_time:.4f} segundos en ejecutarse.")

  ### --------------------------------------------------------------------------
df_salida = pd.DataFrame(salida)


  #Imprimir los valores extraídos (opcional, para verificación)
  # print(f"Fila {index}:")
  # print(f"  consumo_vida_util_actual: {consumo_vida_util_actual}")
  # print(f"  tipo_via_inicio: {tipo_via_inicio}")
  # print(f"  carga_util_teorica: {carga_util_teorica}")
  # print(f"  tipo_trocha: {tipo_trocha}")
  # print(f"  barreras: {barreras}")
  # print(f"  long_tramo: {long_tramo}")
  # print("-" * 20)


  #break




Año Vía Tipo   Carga Util  Carga Util  TNKM      Consumo     Renov   Costo   Costo  Cos 1er Cos 2do  Costo    Veloc   TN x Eje
        Trocha  Teórica     Proyectada            Vida Util   y Mejor Desvio  Fijo   Manten  Manten  Operación
               (TN x mil)  (TN x mil) (x mil)  (x mil)  (%)                                          Infra
0    II  angosta  144          144      152      187000   85%     0     0      0      0        0        0        0        0        0      
1    II  angosta  151          151      160      187151   85%     30    0      5      32       0        11       50       19       2      Comienzo Obra Mejoramiento 1
2    II  angosta  158          158      168      187310   85%     30    0      5      32       0        11       50       19       2      ... Sigue obra Mejoramiento 1
3    II  angosta  166          166      176      187476   85%     30    0      5      32       0        11       50       19       2      ... Sigue obra Mejoramiento 1
4    II  ango

APIError: APIError: [429]: Quota exceeded for quota metric 'Read requests' and limit 'Read requests per minute per user' of service 'sheets.googleapis.com' for consumer 'project_number:522309567947'.

In [ ]:
# NO
# imprimo los valores dentro de "tipo_via_inicio" y "tipo_trocha" y accediendo al dict "costo_mantenimiento_dict" extraer el valor de "crec", "carga_ref", "tope" y al valor correspondiente a "tipo_trocha"

crec = costo_mantenimiento_dict[tipo_via_inicio]['crec']
carga_ref = costo_mantenimiento_dict[tipo_via_inicio]['carga_ref']
tope = costo_mantenimiento_dict[tipo_via_inicio]['tope']
valor_tipo_trocha = costo_mantenimiento_dict[tipo_via_inicio][tipo_trocha]

print(f"Para tipo_via_inicio='{tipo_via_inicio}' y tipo_trocha='{tipo_trocha}':")
print(f"crec: {crec}")
print(f"carga_ref: {carga_ref}")
print(f"tope: {tope}")
print(f"valor_tipo_trocha: {valor_tipo_trocha}")


Para tipo_via_inicio='IV' y tipo_trocha='ancha':
crec: 0.03
carga_ref: 1000
tope: 50
valor_tipo_trocha: 4.5


In [ ]:
# No
########################### SECCCION TEST ######################################

# Pruebas para evaluar la función get_costo_desvio()

# Ejemplo de uso con el diccionario ya definido en el código anterior:
# incremento_cap_dict (asegúrate de que este diccionario esté cargado correctamente)
# carga_util_proyectada (usa un valor de ejemplo o uno calculado en tu loop)

# Supongamos que carga_util_proyectada es 45000 (mayor que el último rango)
carga_ejemplo = 1000
#anio=0
pp1 = get_costo_desvio(carga_ejemplo,99, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1[1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

carga_ejemplo = 3000
#anio=0
pp1 = get_costo_desvio(carga_ejemplo,flagcito, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1[1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

carga_ejemplo = 4200
#anio=0
pp1 = get_costo_desvio(carga_ejemplo,flagcito, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1[1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

# Supongamos que carga_util_proyectada es 25000
carga_ejemplo = 4500
pp1 = get_costo_desvio(carga_ejemplo, flagcito, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1[1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)
#anio=4


# Supongamos que carga_util_proyectada es 5000
carga_ejemplo = 4600
pp1 = get_costo_desvio(carga_ejemplo, flagcito, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1 [1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)


# Supongamos que carga_util_proyectada es 5000
carga_ejemplo = 6500
pp1 = get_costo_desvio(carga_ejemplo, flagcito, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1 [1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

# Supongamos que carga_util_proyectada es 5000
carga_ejemplo = 7500
pp1 = get_costo_desvio(carga_ejemplo, flagcito, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1 [1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

# Supongamos que carga_util_proyectada es 5000
carga_ejemplo = 8500
pp1 = get_costo_desvio(carga_ejemplo, flagcito,  incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1 [1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

# Supongamos que carga_util_proyectada es 5000
carga_ejemplo = 9500
pp1 = get_costo_desvio(carga_ejemplo, flagcito, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1 [1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

# Supongamos que carga_util_proyectada es 5000
carga_ejemplo = 10500
pp1 = get_costo_desvio(carga_ejemplo, flagcito, incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1 [1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

# Supongamos que carga_util_proyectada es 5000
carga_ejemplo = 12000
pp1 = get_costo_desvio(carga_ejemplo, flagcito,  incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1 [1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

# Supongamos que carga_util_proyectada es 5000
carga_ejemplo = 20000
pp1 = get_costo_desvio(carga_ejemplo, flagcito,  incremento_cap_dict)
long_encontrada = pp1[0]
flagcito = pp1 [1]
print(f"Para una carga útil proyectada de {carga_ejemplo}, la 'long' asociada es: {long_encontrada}", flagcito)

Para una carga útil proyectada de 1000, la 'long' asociada es: 0 98
Para una carga útil proyectada de 3000, la 'long' asociada es: 0 98
Para una carga útil proyectada de 4200, la 'long' asociada es: 87.0 0
Para una carga útil proyectada de 4500, la 'long' asociada es: 0 0
Para una carga útil proyectada de 4600, la 'long' asociada es: 0 0
Para una carga útil proyectada de 6500, la 'long' asociada es: 174.0 1
Para una carga útil proyectada de 7500, la 'long' asociada es: 0 1
Para una carga útil proyectada de 8500, la 'long' asociada es: 348.0 2
Para una carga útil proyectada de 9500, la 'long' asociada es: 0 2
Para una carga útil proyectada de 10500, la 'long' asociada es: 0 2
Para una carga útil proyectada de 12000, la 'long' asociada es: 1740.0 -1
Para una carga útil proyectada de 20000, la 'long' asociada es: 0 -1


In [ ]:
## TEST ##

mantenimiento_inicial=4.5
vida_util_x = 75250
x1 = vida_util_x/carga_ref_inicial
x2 = (1+crec_mantenimiento)
x3 = x2**x1
x4 = mantenimiento_inicial*x3
print(x1, x2, x3, x4)
#
costo_manten1 = mantenimiento_inicial*((1+crec_mantenimiento)**(vida_util_x/carga_ref_inicial))

print (costo_manten1)

75.25 1.03 9.247006418166615 41.61152888174977
41.61152888174977


In [ ]:
## TEST ##

vvida_util=93269
costo_manten1 = costo_manten_inicial *((1+crec_manten)**(vvida_util/carga_ref_manten))
print (costo_manten1)
print(50*(valor_primera_renovacion-acum_obra)/valor_primera_renovacion)

70.8806391098418
37.5


In [ ]:
# No
## OJO Exportación de la salida a un excel
df_salida
df_salida.to_excel('salida_modelo_gf.xlsx', index=False)
#print("DataFrame df_salida exportado a 'salida_modelo_gf.xlsx'")

In [ ]:
# NO

# usando df_salida generar la suma, el promedio, el mínimo y el máximo de todos los campos numericos excepto  "id_tramo", "longitud" y "año". Usar id_tramo para agrupar esos campos

# NO
# Seleccionar las columnas numéricas relevantes, excluyendo las especificadas
columnas_numericas = df_salida.select_dtypes(include=np.number).columns.tolist()
#,
columnas_a_excluir = ['id_tramo','longitud', 'año', "tipo_obra", "% Vida útil consumida"] # Excluir también
columnas_a_procesar = [col for col in columnas_numericas if col not in columnas_a_excluir]

# Realizar las agregaciones agrupando por 'id_tramo'
df_agregado = df_salida.groupby('id_tramo')[columnas_a_procesar].agg(['sum', 'mean']) #, 'min', 'max'])

# Opcional: Aplanar el MultiIndex de las columnas para una visualización más sencilla
df_agregado.columns = ['_'.join(col).strip() for col in df_agregado.columns.values]

# Mostrar el DataFrame resultante con las agregaciones
df_agregado = df_agregado.drop(columns=['Velocidad_sum', 'Carga x Eje_sum'])
df_agregado


,Carga útil teórica proyectada por año (q) (tn)_sum,Carga útil teórica proyectada por año (q) (tn)_mean,Carga proyectada por año (q) (tn) - LIMITADA_sum,Carga proyectada por año (q) (tn) - LIMITADA_mean,Carga anual TN.KM_sum,Carga anual TN.KM_mean,Carga equiv. desde última intervención_sum,Carga equiv. desde última intervención_mean,Obra de Renovación_sum,Obra de Renovación_mean,...,Conservación costo fijo anual_sum,Conservación costo fijo anual_mean,Primer Tramo de Mantenimiento_sum,Primer Tramo de Mantenimiento_mean,Segundo Tramo de Mantenimiento_sum,Segundo Tramo de Mantenimiento_mean,Costo operación infraestructura del tramo_sum,Costo operación infraestructura del tramo_mean,Velocidad_mean,Carga x Eje_mean
id_tramo,,,,,,,,,,,,,,,,,,,,,
1,470114,11466.195122,406037,9903.341463,812097.12,19807.246829,5449158,132906.292683,1450.0,35.365854,...,200.0,4.878049,452.88,11.045854,959.57,23.404146,440.0,10.731707,54.268293,21.365854
2,470114,11466.195122,406037,9903.341463,812097.12,19807.246829,5799689,141455.829268,1100.0,26.829268,...,200.0,4.878049,628.54,15.330244,265.55,6.476829,2200.0,53.658537,56.146341,20.414634
3,94008,2292.878049,94008,2292.878049,188053.97,4586.682195,2756258,67225.804878,0.0,0.000000,...,200.0,4.878049,406.15,9.906098,0.00,0.000000,756.0,18.439024,68.292683,21.463415


In [ ]:
# Exporta a gsheet

planilla = gc.open('Resultados Modelo')
#exportar_gsheet(planilla, df_salida, 'Tramos')
exportar_gsheet(planilla, df_agregado, 'Resumen Tramos')

In [ ]:
# NO
## TEST NPV o VAN ##

flujos_obra_mejoramiento = df_salida['Obra de Mejoramiento'].tolist()
npv_obra_mejoramiento = npf.npv(0.1, flujos_obra_mejoramiento)
#print(npv_obra_mejoramiento)
#print(flujos_obra_mejoramiento)



In [ ]:
# No
# usando la lista "salida", crear una lista con la columna 'Obra de Mejoramiento'

obra_mejoramiento_list = [item['Obra de Mejoramiento'] for item in salida]

# Imprimir la nueva lista (opcional)
obra_mejoramiento_list

npv_obra_mejoramiento = npf.npv(0.1, obra_mejoramiento_list)

print(npv_obra_mejoramiento)

212.99774605559725


In [ ]:
# NO utilizando el df_salida, generar una función que anio x anio calcule la carga proyectada multiplicada por la long del tramo para cada id_tramo y que a ese resultado lo divida por la suma de las longitudes de cada id_tramo. Tener en cuenta que son tres tramos diferentes y que cada tramos tiene 40 anios

def calcular_carga_promedio_ponderada(df_salida):
    """
    Calcula la carga proyectada multiplicada por la longitud del tramo para cada
    año y para cada id_tramo, y luego divide este resultado por la suma de
    las longitudes de todos los tramos en ese año.

    Args:
        df_salida: DataFrame con los resultados del modelo, debe contener
                   las columnas 'id_tramo', 'año', 'Carga proyectada por año (q) (tn) - LIMITADA',
                   y 'longitud'.

    Returns:
        Un pandas Series con la carga promedio ponderada por longitud para cada año.
    """
    # Calcular la carga ponderada por longitud para cada fila (cada año de cada tramo)
    df_salida['carga_ponderada'] = df_salida['Carga proyectada por año (q) (tn) - LIMITADA'] * df_salida['longitud']

    # Calcular la suma de las longitudes para cada año
    suma_longitudes_por_anio = df_salida.groupby('año')['longitud'].sum()

    # Calcular la suma de la carga ponderada para cada año
    suma_carga_ponderada_por_anio = df_salida.groupby('año')['carga_ponderada'].sum()

    # Calcular la carga promedio ponderada por año
    carga_promedio_ponderada_por_anio = suma_carga_ponderada_por_anio / suma_longitudes_por_anio

    return carga_promedio_ponderada_por_anio

# Ejemplo de uso (asumiendo que df_salida ya existe y tiene los datos correctos)
# df_salida = pd.DataFrame(salida) # Si df_salida no está definida globalmente

# Calcular la carga promedio ponderada
# carga_promedio_anual = calcular_carga_promedio_ponderada(df_salida)

# Imprimir o usar el resultado
# print(carga_promedio_anual)


In [ ]:
# SI
############### Salida Tramos y Resumen Tramos #################################


# df_agregado agregarle la columna "longitud", "tipo_trocha", "tipo_via_inicio" solo para anio=0 de cada tramo

df_agregado = df_salida.groupby('id_tramo').agg({
    'Carga útil teórica proyectada por año (q) (tn)': ['sum', 'mean'],
    'Carga proyectada por año (q) (tn) - LIMITADA': ['sum', 'mean'],
    'Carga anual TN.KM': ['sum', 'mean'],
    'Carga equiv. desde última intervención': ['sum', 'mean'],
    'Obra de Renovación': ['sum', 'mean'],
    'Obra de Mejoramiento': ['sum', 'mean'],
    'Desvíos a construir': ['sum', 'mean'],
    'Conservación costo fijo anual': ['sum', 'mean'],
    'Primer Tramo de Mantenimiento': ['sum', 'mean'],
    'Segundo Tramo de Mantenimiento': ['sum', 'mean'],
    'Costo operación infraestructura del tramo': ['sum', 'mean'],
    'Velocidad': 'mean',
    'Carga x Eje': 'mean'
})

# Aplanar el MultiIndex de las columnas para una visualización más sencilla
df_agregado.columns = ['_'.join(col).strip() for col in df_agregado.columns.values]

# Obtener las columnas 'longitud', 'tipo_trocha', 'tipo_via_inicio' para anio=0 de cada tramo
df_anio_cero = df_salida[df_salida['año'] == 0][['id_tramo','longitud', 'tipo_trocha', 'tipo_via_inicio']].set_index('id_tramo')
df_anio_cero['id_tramo'] = df_anio_cero.index
cols = ['id_tramo'] + [col for col in df_anio_cero.columns if col != 'id_tramo']
df_anio_cero = df_anio_cero[cols]
#
# Unir las columnas del año 0 al DataFrame agregado

df_agregado = df_anio_cero.join(df_agregado)
#df_agregado.join(df_anio_cero)

#df_anio_cero['id_tramo'] = df_agregado.index

######################## Exportación a un Google Sheet #########################

export_gsheet(df_agregado,"Resumen Tramos", "A2")

print("DataFrame df_salida copied to 'Resumen Tramos' sheet.")

export_gsheet(df_salida,"Tramos", "A2")

print("DataFrame df_salida copied to 'Tramos' sheet.")


DataFrame df_salida copied to 'Resumen Tramos' sheet.
DataFrame df_salida copied to 'Tramos' sheet.


In [ ]:
df_agregado

,longitud,tipo_trocha,tipo_via_inicio,Carga útil teórica proyectada por año (q) (tn)_sum,Carga útil teórica proyectada por año (q) (tn)_mean,Carga proyectada por año (q) (tn) - LIMITADA_sum,Carga proyectada por año (q) (tn) - LIMITADA_mean,Carga anual TN.KM_sum,Carga anual TN.KM_mean,Carga equiv. desde última intervención_sum,...,Conservación costo fijo anual_sum,Conservación costo fijo anual_mean,Primer Tramo de Mantenimiento_sum,Primer Tramo de Mantenimiento_mean,Segundo Tramo de Mantenimiento_sum,Segundo Tramo de Mantenimiento_mean,Costo operación infraestructura del tramo_sum,Costo operación infraestructura del tramo_mean,Velocidad_mean,Carga x Eje_mean
id_tramo,,,,,,,,,,,,,,,,,,,,,
1,2,ancha,IV,470114,11466.195122,406037,9903.341463,812097.12,19807.246829,5665232,...,200.0,4.878049,175.0,4.268293,1017.34,24.813171,440.0,10.731707,56.463415,22.097561


In [ ]:
# NO
# TEST GSHEET
#
spreadsheet = gc.open('Resultados Modelo')
worksheet = spreadsheet.worksheet('Tramos')

RANGO_A_BORRAR = 'A2:U500' # Tu rango como una cadena de texto
worksheet.batch_clear([RANGO_A_BORRAR])
#worksheet.clear(RANGO_A_BORRAR) # Pasa la cadena del rango directamente como el primer argumento.

{'spreadsheetId': '1BXwCFFrku642-9L1F0AXL2HMLi2kM7dM_0E_BhtK7vg',
 'clearedRanges': ['Tramos!A2:U500']}

In [ ]:
# NO
#Suma simple

#def calc_suma_simple_por_anio(df, campo_carga):
    """
    Calcula la suma del campo de carga especificado para cada año, sin dividir
    por la longitud del tramo.

    Args:
        df: pandas DataFrame con los resultados del modelo, debe contener
            las columnas 'año' y el campo especificado para la carga.
        campo_carga: El nombre de la columna que contiene la carga.

    Returns:
        Un pandas Series con la suma del campo de carga por año, indexado por 'año'.
    """
    # Agrupar por 'año' y sumar el campo de carga
    suma_simple_por_anio = df.groupby('año')[campo_carga].sum()

    return suma_simple_por_anio

# Ejemplo de uso con el DataFrame df_salida y el campo 'Carga proyectada por año (q) (tn) - LIMITADA'
suma_carga_anual_simple = calc_suma_simple_por_anio(df_salida, 'Carga anual TN.KM')

# Imprimir el resultado
print("\nSuma de 'Carga proyectada por año (q) (tn) - LIMITADA' por año:")
print(suma_carga_anual_simple)



Suma de 'Carga proyectada por año (q) (tn) - LIMITADA' por año:
año
0    10000.0
1    10500.0
Name: Carga anual TN.KM, dtype: float64


In [ ]:
## TEST funciones de salida

# Ejemplo de uso con el DataFrame df_salida y el campo 'Carga anual TN.KM'
suma_carga_anual_simple = calc_suma_simple_por_anio(df_salida, 'Carga anual TN.KM')

# Imprimir el resultado
print("\nSuma de 'Carga anual TN.KM' por año (hasta año 40 con 0s para años sin datos):")
suma_carga_anual_simple


In [ ]:
### NO

def calcular_suma_producto_carga_npv(df, campo_carga):
  """
  Calcula la suma producto de la 'Carga útil teórica proyectada por año'
  multiplicado por la 'longitud' para cada 'id_tramo', considerando años > 0.

  Args:
    df: pandas DataFrame con los resultados del modelo, debe contener
        las columnas 'id_tramo', 'año', el campo especificado para la carga,
        y 'longitud'.
    campo_carga: El nombre de la columna que contiene la carga proyectada
                 (por defecto 'Carga útil teórica proyectada por año (q) (tn)').

  Returns:
    Un pandas Series con la suma producto para cada 'id_tramo',
    indexado por 'id_tramo'.
  """
  longitud_tramos = df_salida[df_salida['año'] == 0].groupby('año')['longitud'].sum().iloc[0]
  print (longitud_tramos)

  # Filtrar el DataFrame para incluir solo los años mayores a 0
  df_filtrado = df.copy()
  #df[df['año'] > 0].copy()

  # Calcular el producto de la carga y la longitud para los años filtrados
  df_filtrado['carga_x_longitud'] = df_filtrado[campo_carga] * df_filtrado['longitud']

  # Calcular la suma de 'carga_x_longitud' agrupando por 'id_tramo'
  suma_producto_por_anio = df_filtrado.groupby('año')['carga_x_longitud'].sum()/longitud_tramos

  return round(suma_producto_por_anio,2)

In [ ]:
# NO
# función "calcular_suma_producto_carga_npv" agregarle que calcule el valor presente neto con una tasa de "0,1" y que devuelva ese valor

def calcular_suma_producto_carga_npv(df, campo_carga, tasa_descuento=0.1):
  """
  Calcula la suma producto de la 'Carga útil teórica proyectada por año'
  multiplicado por la 'longitud' para cada 'id_tramo', considerando años > 0,
  y calcula el Valor Presente Neto (NPV) de esta serie anual.

  Args:
    df: pandas DataFrame con los resultados del modelo, debe contener
        las columnas 'id_tramo', 'año', el campo especificado para la carga,
        y 'longitud'.
    campo_carga: El nombre de la columna que contiene la carga proyectada
                 (por defecto 'Carga útil teórica proyectada por año (q) (tn)').
    tasa_descuento: La tasa de descuento a utilizar para el cálculo del NPV (por defecto 0.1).

  Returns:
    Un diccionario con dos claves:
      'suma_producto_anual': Un pandas Series con la suma producto para cada año.
      'npv': El Valor Presente Neto de la serie suma_producto_anual.
  """
  # Obtener la suma total de longitudes de tramo en el año 0
  longitud_tramos = df[df['año'] == 0]['longitud'].sum()

  # Filtrar el DataFrame para incluir solo los años mayores a 0
  df_filtrado = df
   #[df['año'] > 0].copy()

  # Calcular el producto de la carga y la longitud para los años filtrados
  df_filtrado['carga_x_longitud'] = df_filtrado[campo_carga] * df_filtrado['longitud']

  # Calcular la suma de 'carga_x_longitud' agrupando por 'año' y dividir por la suma total de longitudes
  # para obtener la carga promedio ponderada por longitud para cada año
  suma_producto_por_anio = df_filtrado.groupby('año')['carga_x_longitud'].sum()

  # Calcular el NPV de la serie de suma_producto_por_anio
  # numpy_financial.npv espera los flujos de efectivo comenzando desde el período 1
  npv_valor = npf.npv(tasa_descuento, suma_producto_por_anio)

  return {
      'suma_producto_anual': round(suma_producto_por_anio, 2),
      'npv': round(npv_valor, 2)
  }

# Ejemplo de uso con el DataFrame df_salida y el campo 'Carga útil teórica proyectada por año (q) (tn)'
resultado_npv = calcular_suma_producto_carga_npv(df_salida, 'Obra de Renovación', tasa_descuento=0.1)
print (resultado_npv['suma_producto_anual'])
print (resultado_npv['npv'])

# # Imprimir el resultado
# print("Suma producto de 'Carga útil teórica proyectada por año' por 'longitud' (promedio ponderado) y su NPV:")
# print("Suma producto anual:", resultado_npv['suma_producto_anual'])
# print("NPV:", resultado_npv['npv'])

# # Si quieres usar la columna 'Carga proyectada por año (q) (tn) - LIMITADA' en su lugar
# resultado_npv_limitada = calcular_suma_producto_carga_npv(df_salida, 'Carga proyectada por año (q) (tn) - LIMITADA', tasa_descuento=0.1)
# print("\nSuma producto de 'Carga proyectada por año (q) (tn) - LIMITADA' por 'longitud' (promedio ponderado) y su NPV:")
# print("Suma producto anual (Limitada):", resultado_npv_limitada['suma_producto_anual'])
# print("NPV (Limitada):", resultado_npv_limitada['npv'])


año
0       0.0
1       0.0
2       0.0
3       0.0
4       0.0
5       0.0
6       0.0
7       0.0
8       0.0
9       0.0
10    725.0
11    725.0
12    725.0
13    725.0
14      0.0
15      0.0
16      0.0
17      0.0
18      0.0
19      0.0
20      0.0
21      0.0
22    550.0
23    550.0
24    550.0
25    550.0
26      0.0
27      0.0
28      0.0
29      0.0
30      0.0
31      0.0
32      0.0
33      0.0
34      0.0
35      0.0
36      0.0
37      0.0
38      0.0
39      0.0
40      0.0
Name: carga_x_longitud, dtype: float64
1210.23


In [ ]:
# NO
# a esta columna "resultado_npv['suma_producto_anual']" recien generada copiarla en la planilla "Resultados Modelo" en la fila 13 desde la columna E en adelante

spreadsheet = gc.open('Resultados Modelo')
worksheet_resultados = spreadsheet.worksheet('Copia de Consolidado')

# Convertir la Serie de suma_producto_anual a una lista de listas para actualizar la hoja de cálculo
# La lista debe contener los encabezados si se van a escribir.
# Como queremos copiar desde la columna E, solo necesitamos los valores.
# También hay que asegurar que los datos están en el orden correcto por año si es necesario.
# En este caso, resultado_npv['suma_producto_anual'] ya es una Serie indexada por año.
# Asumimos que la fila 13 es para "Obra de Renovación (Total en millones de USD) Suma de los Tramos"
# y que queremos colocar los valores anuales a partir de la columna E (índice 4).

# Obtener la serie de suma_producto_anual
#suma_producto_serie = resultado_npv['suma_producto_anual']

resultado_npv = calc_suma_producto(df_salida, 'Obra de Renovación', 1, 1)
# print (resultado_npv['suma_producto_anual'])
# print (resultado_npv['npv'])

# Convertir la serie a una lista de listas, donde cada lista interna es un año's valor
# Necesitamos los años desde el 0 hasta el final
# Asegurarse de que la serie cubre todos los años que quieres escribir (0 a duracion_anios_negocio)
# Si la serie no contiene el año 0, podrías necesitar añadirlo con un valor de 0.
# Ejemplo: crear una lista con los valores en el orden de los años
#valores_a_escribir = [suma_producto_serie.get(anio, 0) for anio in range(duracion_anios_negocio + 1)]

# Convertir la lista simple a una lista de listas para gspread (gspread.update espera una lista de listas)
# Cada elemento de la lista principal es una fila. Queremos escribir una sola fila.
#output_row = resultado_npv['suma_producto_anual']

# Especificar la celda donde comienza la escritura.
# Fila 13 (índice 12) y Columna E (índice 4).
#start_cell = 'E13'

# Actualizar la hoja de cálculo.
# gspread.update(range_name, values)
#worksheet_resultados.update(output_row, "E13:AS13")

print(valores_a_escribir)
print(f"Datos de 'suma_producto_anual' copiados a la hoja 'Resultados Modelo' comenzando en {start_cell}")


In [ ]:
df_salida

,id_tramo,longitud,tipo_trocha,tipo_via_inicio,año,Carga útil teórica proyectada por año (q) (tn),Carga proyectada por año (q) (tn) - LIMITADA,Carga anual TN.KM,Carga equiv. desde última intervención,% Vida útil consumida,...,Obra de Mejoramiento,Desvíos a construir,Conservación costo fijo anual,Primer Tramo de Mantenimiento,Segundo Tramo de Mantenimiento,Costo operación infraestructura del tramo,Velocidad,Carga x Eje,tipo_obra,marca
0,1434,2.36,ancha,II,0,6065,6065,14313.49,114400,0.52,...,0.0,0.0,0.0,0.00,0.0,0.00,0,0,0,
1,1434,2.36,ancha,II,1,6368,6368,15029.16,120768,0.55,...,32.5,144.0,5.0,19.72,0.0,9.32,60,22,2,Comienzo Obra Mejoramiento 1
2,1434,2.36,ancha,II,2,6686,6686,15780.62,127454,0.58,...,32.5,0.0,5.0,21.06,0.0,9.32,60,22,2,... Sigue obra Mejoramiento 1
3,1434,2.36,ancha,II,3,7021,7021,16569.65,134476,0.61,...,32.5,0.0,5.0,22.57,0.0,9.32,60,22,2,... Sigue obra Mejoramiento 1
4,1434,2.36,ancha,II,4,7372,7372,17398.13,141848,0.64,...,32.5,0.0,5.0,24.27,0.0,9.32,60,22,2,... Sigue obra Mejoramiento 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72893,401,1.00,angosta,I,36,2,2,2.95,57,0.00,...,0.0,0.0,5.0,0.00,5.0,12.00,70,22,0,
72894,401,1.00,angosta,I,37,3,3,3.04,60,0.00,...,0.0,0.0,5.0,0.00,5.0,12.00,70,22,0,
72895,401,1.00,angosta,I,38,3,3,3.13,63,0.00,...,0.0,0.0,5.0,0.00,5.0,12.00,70,22,0,
72896,401,1.00,angosta,I,39,3,3,3.22,66,0.00,...,0.0,0.0,5.0,0.00,5.0,12.00,70,22,0,


In [ ]:
# prompt: tengo esto "
# longitud_tramos = df_salida[df_salida['año'] == 0].groupby('año')['longitud'].sum().iloc[0]
# worksheet.update([longitud_tramos], "B1")"

longitud_tramos = df_salida[df_salida['año'] == 0].groupby('año')['longitud'].sum().iloc[0]
worksheet.update([[longitud_tramos]], "B1")


In [ ]:
# SI
# SALIDA HORIZONTAL
###################### Carga del GSheet de Salida ##############################

### Son varias filas
### Implementar el cero en cada relleno ###

### Salida a la planilla gsheet ###

# Assuming 'Resultados Modelo' is the spreadsheet name
spreadsheet = gc.open('Resultados Modelo')

# Assuming 'Consolidado' is the sheet name where you want to paste the data
worksheet = spreadsheet.worksheet('Copia de Consolidado')

longitud_tramos = df_salida[df_salida['año'] == 0].groupby('año')['longitud'].sum().iloc[0]
worksheet.update([[longitud_tramos]], "B1")

resultado = calc_suma_producto(df_salida, 'Obra de Renovación', 1,1)
#print (resultado_npv['suma_producto_anual'])
#print (resultado_npv['npv'])

#columna_vertical = resultado_npv['suma_producto_anual']

# Convert the Series to a list and skip the first value
fila_horizontal = resultado.tolist()
#print (fila_horizontal)

# Define the starting cell (e.g., E1) for the horizontal row
# gspread uses A1 notation, so E1 refers to the first row, fifth column
start_cell = 'E12'

# Update the sheet with the horizontal row
# gspread expects a list of lists for updating. Since it's a single row,
# the outer list contains one inner list with the data.
worksheet.update([fila_horizontal], start_cell)

#print(f"Columna 'Carga proyectada por año (q) (tn) - LIMITADA' (sin el primer valor) copiada a la fila horizontal comenzando en '{start_cell}' de la hoja 'Consolidado'.")

resultado = calc_suma_producto(df_salida, 'Carga proyectada por año (q) (tn) - LIMITADA', 0,0)
fila_horizontal = resultado.tolist()
start_cell = 'D8'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_simple_por_anio(df_salida, 'Carga anual TN.KM')
fila_horizontal = resultado.tolist()
start_cell = 'D9'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, '% Vida útil consumida', 0,0)
fila_horizontal = resultado.tolist()
start_cell = 'D10'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, 'Obra de Mejoramiento', 1,1)
fila_horizontal = resultado.tolist()
start_cell = 'E13'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, 'Desvíos a construir', 1,1)
fila_horizontal = resultado.tolist()
start_cell = 'E14'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, 'Conservación costo fijo anual', 1,1)
fila_horizontal = resultado.tolist()
start_cell = 'E17'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, 'Primer Tramo de Mantenimiento', 1,1)
fila_horizontal = resultado.tolist()
start_cell = 'E18'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, 'Segundo Tramo de Mantenimiento', 1,1)
fila_horizontal = resultado.tolist()
start_cell = 'E19'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, 'Costo operación infraestructura del tramo', 1,1)
fila_horizontal = resultado.tolist()
start_cell = 'E21'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, 'Velocidad', 1,0)
fila_horizontal = resultado.tolist()
start_cell = 'E27'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_suma_producto(df_salida, 'Carga x Eje', 1,0)
fila_horizontal = resultado.tolist()
start_cell = 'E28'
worksheet.update([fila_horizontal], start_cell)

resultado = calc_min_carga_por_anio(df_salida, 'Carga x Eje')
fila_horizontal = resultado.tolist()
start_cell = 'E29'
worksheet.update([fila_horizontal], start_cell)


{'spreadsheetId': '1BXwCFFrku642-9L1F0AXL2HMLi2kM7dM_0E_BhtK7vg',
 'updatedRange': "'Copia de Consolidado'!E29:AR29",
 'updatedRows': 1,
 'updatedColumns': 40,
 'updatedCells': 40}

In [ ]:
# SI
# SALIDA VERTICAL
###################### Carga del GSheet de Salida ##############################

### Son varias filas
### Implementar el cero en cada relleno ###

### Salida a la planilla gsheet ###

# Assuming 'Resultados Modelo' is the spreadsheet name
spreadsheet = gc.open('Resultados Modelo')

# Assuming 'Consolidado' is the sheet name where you want to paste the data
worksheet = spreadsheet.worksheet('consolidado2')

resultado = calc_suma_producto(df_salida, 'Carga proyectada por año (q) (tn) - LIMITADA', 0,0)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'A2'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_simple_por_anio(df_salida, 'Carga anual TN.KM')
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'B2'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, '% Vida útil consumida', 0, 0)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'C2'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Obra de Renovación', 1,1)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'D3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Obra de Mejoramiento', 1,1)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'E3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Desvíos a construir', 1,1)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'F3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Conservación costo fijo anual', 1,1)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'G3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Primer Tramo de Mantenimiento', 1,1)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'H3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Segundo Tramo de Mantenimiento', 1,1)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'I3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Costo operación infraestructura del tramo', 1,1)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'J3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Velocidad', 1, 0)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'L3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_suma_producto(df_salida, 'Carga x Eje', 1, 0)
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'M3'
worksheet.update(columna_vertical, start_cell)

resultado = calc_min_carga_por_anio(df_salida, 'Carga x Eje')
columna_vertical = [[value] for value in resultado.tolist()]
start_cell = 'N3'
worksheet.update(columna_vertical, start_cell)




{'spreadsheetId': '1BXwCFFrku642-9L1F0AXL2HMLi2kM7dM_0E_BhtK7vg',
 'updatedRange': 'consolidado2!N3:N42',
 'updatedRows': 40,
 'updatedColumns': 1,
 'updatedCells': 40}

In [ ]:
df_salida

,id_tramo,longitud,tipo_trocha,tipo_via_inicio,año,Carga útil teórica proyectada por año (q) (tn),Carga proyectada por año (q) (tn) - LIMITADA,Carga anual TN.KM,Carga equiv. desde última intervención,% Vida útil consumida,...,Obra de Mejoramiento,Desvíos a construir,Conservación costo fijo anual,Primer Tramo de Mantenimiento,Segundo Tramo de Mantenimiento,Costo operación infraestructura del tramo,Velocidad,Carga x Eje,tipo_obra,marca
0,1,2,ancha,IV,0,5000,5000,10000.0,70000,0.70,...,0.00,0.0,0.0,0.00,0.0,0.0,0,0,0,
1,1,2,ancha,IV,1,5250,5250,10500.0,75250,0.75,...,33.75,87.0,5.0,41.61,0.0,11.0,55,20,2,Comienzo Obra Mejoramiento 1


In [ ]:

df_salida.to_csv('df_salida.csv', index=False) # index=False evita guardar el índice del DataFrame como una columna

print("DataFrame df_salida guardado en 'df_salida.csv'")
